# Multimodal RAG — Adani Green Energy Annual Report
### Updated for LangChain 2026 (v1.x stack)

**What changed from older code:**
- `init_chat_model` replaced with direct `ChatOpenAI` / `ChatGroq` imports
- `FAISS.from_embeddings` updated signature (langchain-community 0.4.x)
- No deprecated `LLMChain` or `ConversationalRetrievalChain` — uses plain `.invoke()`
- `langchain_openai.ChatOpenAI` instead of old `langchain.chat_models`
- All imports from the correct sub-packages (`langchain_core`, `langchain_openai`, `langchain_groq`)

**Folder structure expected:**
```
RAGs/Multimodal_RAG/
    pdf/        <-- adani_green_annual_report.pdf goes here
    docs/       <-- multimodal_rag_guide.md
    notebook/   <-- this file (adani_multimodal_rag.ipynb)
    .env        <-- or project root .env
```


## Step 0 — Install / Upgrade All Packages

In [ ]:
# Run this cell once, then restart the kernel before continuing.
# Uses the exact 2026-current versions of every package.

import sys

packages = [
    "pymupdf==1.27.2.3",          # PDF parsing  (import fitz)
    "langchain==1.3.1",
    "langchain-core==1.4.0",
    "langchain-community==0.4.1",
    "langchain-openai==1.2.1",
    "langchain-groq==1.1.2",
    "faiss-cpu==1.13.2",          # vector store
    "transformers==4.51.3",       # CLIP
    "torch",                      # CLIP backend
    "Pillow",                     # image handling
    "python-dotenv",              # .env loader
    "numpy",
    "scikit-learn",
    "openai",                     # underlying client for ChatOpenAI
]

import subprocess
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("All packages installed. RESTART THE KERNEL NOW before running the next cells.")


## Step 1 — Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────
import os
import io
import base64

# ── Numerical / ML ────────────────────────────────────────────
import numpy as np
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel   # unified text+image embeddings

# ── PDF parsing ───────────────────────────────────────────────
import fitz                                          # PyMuPDF

# ── LangChain 2026 — correct import paths ─────────────────────
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter   # moved to langchain_text_splitters
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore   # needed for FAISS manual build

# ── Environment ───────────────────────────────────────────────
from dotenv import load_dotenv

print("All imports successful.")


## Step 2 — Load API Keys and Initialise CLIP

In [ ]:
# ── Load .env (looks upward from current folder automatically) ─
load_dotenv(dotenv_path=r"C:\Users\admin\Desktop\New_GenAI\GenAI\.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY

print("OpenAI key loaded:", "YES" if OPENAI_API_KEY else "MISSING — check .env")
print("Groq   key loaded:", "YES" if GROQ_API_KEY   else "MISSING — check .env")

# ── CLIP model ─────────────────────────────────────────────────
# CLIP maps text AND images into the same 512-dim vector space.
# This is the core that makes cross-modal retrieval possible.
print("\nLoading CLIP model (downloads ~600 MB on first run)...")
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()   # inference only — no gradients needed

print("CLIP ready.")


## Step 3 — CLIP Embedding Functions

In [ ]:
def embed_image(image_input) -> np.ndarray:
    """
    Produce a 512-dim L2-normalised CLIP embedding for an image.
    Accepts a file path string or a PIL.Image object.
    """
    img = (
        Image.open(image_input).convert("RGB")
        if isinstance(image_input, str)
        else image_input.convert("RGB")
    )
    inputs = clip_processor(images=img, return_tensors="pt")
    with torch.no_grad():
        feat = clip_model.get_image_features(**inputs)
        feat = feat / feat.norm(dim=-1, keepdim=True)   # L2 normalise
    return feat.squeeze().cpu().numpy()


def embed_text(text: str) -> np.ndarray:
    """
    Produce a 512-dim L2-normalised CLIP embedding for a text string.
    CLIP truncates at 77 tokens — keep chunks short accordingly.
    """
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    )
    with torch.no_grad():
        feat = clip_model.get_text_features(**inputs)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze().cpu().numpy()


print("Embedding functions defined.")


## Step 4 — Set PDF Path and Initialise Storage

In [ ]:
# ── Point this to your Adani Green annual report PDF ──────────
PDF_PATH = r"C:\Users\admin\Desktop\New_GenAI\GenAI\RAGs\Multimodal_RAG\pdf\adani_green_annual_report.pdf"

# Verify the file exists before proceeding
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found at:\n  {PDF_PATH}\n"
        "Place adani_green_annual_report.pdf in the pdf/ folder."
    )

# ── Storage ────────────────────────────────────────────────────
all_docs         = []    # List[Document] — text chunks + image placeholders
all_embeddings   = []    # parallel List[np.ndarray] — one 512-dim vector per doc
image_data_store = {}    # {image_id: base64_png_str}  — raw images for GPT-4V

# ── Text splitter ──────────────────────────────────────────────
# chunk_size=400 keeps text within CLIP's 77-token window comfortably
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)

print(f"PDF path set: {PDF_PATH}")


## Step 5 — Parse PDF: Extract Text Chunks and Images

In [ ]:
pdf_doc = fitz.open(PDF_PATH)
print(f"Opened: {PDF_PATH}  ({len(pdf_doc)} pages)\n")

for page_num, page in enumerate(pdf_doc):

    # ── TEXT ──────────────────────────────────────────────────
    raw_text = page.get_text()
    if raw_text.strip():
        # Wrap in a Document so the splitter can handle metadata
        base_doc    = Document(
            page_content=raw_text,
            metadata={"page": page_num, "type": "text", "source": PDF_PATH}
        )
        text_chunks = splitter.split_documents([base_doc])

        for chunk in text_chunks:
            emb = embed_text(chunk.page_content)
            all_embeddings.append(emb)
            all_docs.append(chunk)

    # ── IMAGES ────────────────────────────────────────────────
    # get_images(full=True) returns all image objects on the page.
    # Each image is: (xref, smask, width, height, bpc, colorspace,
    #                  alt_colorspace, name, filter, referencer)
    for img_idx, img_info in enumerate(page.get_images(full=True)):
        try:
            xref        = img_info[0]
            base_image  = pdf_doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert bytes → PIL for CLIP
            pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Skip very small images (logos, bullets, decorative icons)
            if pil_img.width < 100 or pil_img.height < 100:
                continue

            # Unique ID for this image
            image_id = f"page_{page_num}_img_{img_idx}"

            # Save base64 PNG — this is what GPT-4 Vision receives
            buf = io.BytesIO()
            pil_img.save(buf, format="PNG")
            image_data_store[image_id] = base64.b64encode(buf.getvalue()).decode()

            # CLIP embedding (image → same 512-dim space as text)
            emb = embed_image(pil_img)
            all_embeddings.append(emb)

            # Placeholder Document so the image participates in retrieval
            img_doc = Document(
                page_content=f"[Chart/Image: {image_id}]",
                metadata={
                    "page":     page_num,
                    "type":     "image",
                    "image_id": image_id,
                    "source":   PDF_PATH,
                }
            )
            all_docs.append(img_doc)

        except Exception as err:
            print(f"  Skipped image {img_idx} on page {page_num}: {err}")

pdf_doc.close()

text_count  = sum(1 for d in all_docs if d.metadata["type"] == "text")
image_count = len(image_data_store)
print(f"Parsed complete.")
print(f"  Text chunks : {text_count}")
print(f"  Images      : {image_count}")
print(f"  Total docs  : {len(all_docs)}")


## Step 6 — Build FAISS Vector Index

In [ ]:
# ── Convert embeddings list → numpy array ─────────────────────
embeddings_array = np.array(all_embeddings, dtype=np.float32)
print(f"Embeddings matrix shape: {embeddings_array.shape}")
# Expected: (N, 512)  where N = text_chunks + images

# ── Build FAISS index via from_embeddings ──────────────────────
# langchain-community 0.4.x: from_embeddings expects
#   text_embeddings: List[Tuple[str, List[float]]]
#   embedding:       an Embeddings object OR None when pre-computed
#   metadatas:       List[dict]
#
# We pass embedding=None because we already have CLIP vectors.
# NOTE: a dummy FakeEmbeddings wrapper is required in v0.4.x
#       when embedding=None to avoid an internal assertion error.

from langchain_community.embeddings.fake import FakeEmbeddings

# FakeEmbeddings must match our vector dimension (512)
fake_emb = FakeEmbeddings(size=512)

vector_store = FAISS.from_embeddings(
    text_embeddings=[
        (doc.page_content, emb.tolist())
        for doc, emb in zip(all_docs, embeddings_array)
    ],
    embedding=fake_emb,
    metadatas=[doc.metadata for doc in all_docs],
)

print(f"FAISS index built: {embeddings_array.shape[0]} vectors "
      f"in {embeddings_array.shape[1]}-dim CLIP space.")


## Step 7 — Choose Your LLM

Run **either** Cell 7a (OpenAI) **or** Cell 7b (Groq). Do not run both.


In [ ]:
# ── Option A: OpenAI GPT-4.1 Vision (recommended) ─────────────
# langchain-openai 1.2.1 — ChatOpenAI is the correct class
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1",           # vision-capable, latest GPT-4.1
    api_key=OPENAI_API_KEY,
    temperature=0,
    max_tokens=1500,
)

print(f"LLM ready (OpenAI): {llm.model_name}")


In [ ]:
# ── Option B: Groq (free tier, fast inference) ────────────────
# langchain-groq 1.1.2 — ChatGroq is the correct class
# Vision support: use 'meta-llama/llama-4-scout-17b-16e-instruct'
# Text-only:      use 'llama-3.3-70b-versatile'
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",  # vision capable
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=1500,
)

print(f"LLM ready (Groq): {llm.model_name}")


## Step 8 — Retrieval Function

In [ ]:
def retrieve_multimodal(query: str, k: int = 5) -> list[Document]:
    """
    Embed the query with CLIP text encoder, then search the FAISS index.

    Because CLIP places text and image embeddings in the same vector space,
    a text query can return both relevant text chunks AND relevant chart images
    ranked together by cosine similarity.

    Returns a list of Document objects (mix of text and image types).
    """
    query_vec = embed_text(query)

    # similarity_search_by_vector is available in langchain-community 0.4.x
    results = vector_store.similarity_search_by_vector(
        embedding=query_vec.tolist(),
        k=k,
    )
    return results


# Quick test
test_results = retrieve_multimodal("revenue growth", k=3)
print(f"Test retrieval returned {len(test_results)} docs:")
for r in test_results:
    print(f"  type={r.metadata['type']}  page={r.metadata['page']}  "
          f"preview={r.page_content[:60].strip()!r}")


## Step 9 — Multimodal Message Builder

In [ ]:
def build_multimodal_message(query: str, docs: list[Document]) -> HumanMessage:
    """
    Assemble a HumanMessage that contains:
      - The user's question
      - Retrieved text excerpts as plain text
      - Retrieved chart images as base64-encoded PNG blocks

    This format is accepted by GPT-4.1, GPT-4V, Claude 3, and Llava.
    langchain_core.messages.HumanMessage supports a list for `content`
    as of langchain-core 1.x — no deprecated schema needed.
    """
    content_blocks = []

    # ── Question header ────────────────────────────────────────
    content_blocks.append({
        "type": "text",
        "text": f"Question: {query}\n\nBelow is the retrieved context from the document.\n",
    })

    # ── Separate text vs image docs ────────────────────────────
    text_docs  = [d for d in docs if d.metadata.get("type") == "text"]
    image_docs = [d for d in docs if d.metadata.get("type") == "image"]

    # ── Text context ───────────────────────────────────────────
    if text_docs:
        text_block = "\n\n".join(
            f"[Page {d.metadata['page']}]:\n{d.page_content}"
            for d in text_docs
        )
        content_blocks.append({
            "type": "text",
            "text": f"TEXT EXCERPTS:\n{text_block}\n",
        })

    # ── Image context ──────────────────────────────────────────
    for doc in image_docs:
        img_id = doc.metadata.get("image_id")
        if img_id and img_id in image_data_store:
            content_blocks.append({
                "type": "text",
                "text": f"\nCHART/IMAGE from page {doc.metadata['page']}:",
            })
            content_blocks.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[img_id]}",
                    "detail": "high",    # GPT-4V detail level: 'low' or 'high'
                },
            })

    # ── Final instruction ──────────────────────────────────────
    content_blocks.append({
        "type": "text",
        "text": (
            "\nUsing ONLY the text excerpts and chart images provided above, "
            "answer the question accurately. If the answer is in a chart, "
            "describe what the chart shows."
        ),
    })

    return HumanMessage(content=content_blocks)


print("Message builder defined.")


## Step 10 — Full RAG Pipeline

In [ ]:
def multimodal_rag(query: str, k: int = 5, verbose: bool = True) -> str:
    """
    End-to-end Multimodal RAG pipeline:

    1. Embed query with CLIP
    2. Retrieve top-k docs from FAISS (text + images)
    3. Build a multimodal HumanMessage
    4. Call the vision LLM via .invoke()  [no deprecated chains]
    5. Return the answer string

    Uses langchain_core HumanMessage.invoke() — works with
    ChatOpenAI 1.2.x and ChatGroq 1.1.x.
    """
    # Step 1+2: Retrieve
    retrieved = retrieve_multimodal(query, k=k)

    if verbose:
        print(f"Query   : {query}")
        print(f"Retrieved {len(retrieved)} docs:")
        for d in retrieved:
            t = d.metadata.get("type")
            p = d.metadata.get("page")
            if t == "text":
                print(f"  [TEXT  p.{p}] {d.page_content[:70].strip()!r}")
            else:
                print(f"  [IMAGE p.{p}] {d.metadata.get('image_id')}")

    # Step 3: Build message
    message = build_multimodal_message(query, retrieved)

    # Step 4: LLM call — plain .invoke(), no LLMChain, no deprecated wrappers
    response = llm.invoke([message])

    # Step 5: Return content
    return response.content


print("Pipeline ready. Run the demo queries in the next cell.")


## Step 11 — Demo Queries on the Adani Green PDF

In [ ]:
# ── Demo queries covering text, image, and mixed retrieval ─────
queries = [
    # Text retrieval
    "What is the revenue growth of Adani Green Energy from FY20 to FY26?",
    "What are the key risks of investing in Adani Green Energy?",
    "Explain the debt situation and interest coverage ratio.",

    # Image retrieval (CLIP aligns these text queries to chart images)
    "What does the operating profit margin chart show?",
    "Describe the shareholding pattern pie chart.",
    "What does the cash flow waterfall chart indicate for FY26?",

    # Mixed — needs both text and chart
    "Why was Q3 FY26 a weak quarter despite an 86% operating margin?",
    "How does Adani Green compare to NTPC by market capitalisation?",
]

# Run all queries and print answers
for query in queries:
    print("=" * 70)
    answer = multimodal_rag(query, k=5, verbose=True)
    print(f"\nANSWER:\n{answer}")
    print()


## Step 12 — Ask Your Own Question

In [ ]:
# Change the query string below and run this cell to ask anything
# about the Adani Green Energy annual report PDF.

my_query = "What is the total asset base of Adani Green Energy in FY26?"

answer = multimodal_rag(my_query, k=5, verbose=True)
print(f"\nANSWER:\n{answer}")
